<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/05_Recommendation_Systems/01_Persona_Aware_Recommender_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: Persona-Aware Hybrid Basket Recommender

## Research Objective
To investigate whether customer-segment-specific association rules can improve next-item basket recommendations compared with non-personalized popularity-based baselines.

**H0:** Segment-aware recommendation does not improve ranking performance over baseline methods.
**H1:** Segment-aware recommendation improves ranking performance over baseline methods.

## Step 1: Solving Information Leakage (The Temporal Split)
If a recommendation engine is tested on the exact same data used to discover its underlying association rules, the evaluation is academically invalid (Information Leakage).

To simulate a real-world deployment, we will construct a strict temporal split:
*   **Training Data (First 80% of chronological transactions):** Used to learn shopping behavior, calculate popularity baselines, and mine persona-specific association rules.
*   **Testing Data (Final 20% of chronological transactions):** Used exclusively to build "hidden-item" baskets to evaluate our recommendation algorithms using `Precision@K`, `Recall@K`, and `NDCG@K`.

In [6]:
# Install the Dunnhumby Complete Journey dataset package
!pip install completejourney_py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 59.4 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
import ast
from completejourney_py import get_data

# 1. Fetch the raw data using your library
data = get_data()
transactions = data["transactions"]
products = data["products"]

# 2. Loading the Master Cluster labels from Phase 3
clusters = pd.read_csv('master_customers_fully_clustered.csv')

# 3. Map the cluster labels and product categories onto the raw transactions
basket_data = transactions.merge(clusters[['household_id', 'Hierarchical_Cluster']], on='household_id', how='inner')
basket_data = basket_data.merge(products[['product_id', 'product_category']], on='product_id', how='left')

# 4. Sort the entire dataset chronologically to simulate time
basket_data['transaction_timestamp'] = pd.to_datetime(basket_data['transaction_timestamp'])
basket_data = basket_data.sort_values('transaction_timestamp')

# 5. Execute the 80/20 Temporal Split
split_idx = int(len(basket_data) * 0.8)
train_data = basket_data.iloc[:split_idx].copy()
test_data = basket_data.iloc[split_idx:].copy()

print("✅ Temporal Split Complete!")
print(f"Training Data (Learning): {len(train_data):,} individual item scans")
print(f"Testing Data (Evaluation): {len(test_data):,} individual item scans")

✅ Temporal Split Complete!
Training Data (Learning): 663,080 individual item scans
Testing Data (Evaluation): 165,770 individual item scans
